# Part 3: Sentiment classification

Compare a 20,000-review trial with a full-corpus run. Both use an 80/20 stratified split and fit TF-IDF only on training text. The full run loads and cleans all 50,000 original rows, then excludes empty, conflicting, and duplicate cleaned reviews.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import sentiment_classifier as sc
sc.prepare_resources()
df = sc.load_reviews()
cleaned, usable = sc.clean_corpus(df)
results = ROOT / "results"
results.mkdir(exist_ok=True)

Reviews cleaned to an empty string: 0


Conflicting cleaned texts excluded: 0
Repeated cleaned texts removed: 426
Usable unique reviews: 49574


## Trial: 20,000 reviews

In [2]:
from sklearn.model_selection import train_test_split
sample, _ = train_test_split(usable, train_size=20000, stratify=usable.sentiment, random_state=42)
sample_model, sample_metrics = sc.evaluate(sample, "sample_20000", results)


EXPERIMENT: sample_20000 train: 16000 test: 4000
Accuracy: 0.88275 | assignment baseline: 0.500 | improvement: 0.38275000000000003
Training-majority baseline on test: 0.50175
              precision    recall  f1-score   support

    negative     0.8818    0.8831    0.8824      1993
    positive     0.8837    0.8824    0.8831      2007

    accuracy                         0.8828      4000
   macro avg     0.8827    0.8828    0.8827      4000
weighted avg     0.8828    0.8828    0.8828      4000

Confusion matrix: rows=true, columns=predicted; negative, positive
 [[1760  233]
 [ 236 1771]]

20 most positive terms:
                   term  positive_minus_negative_log_probability
     highly recommended                                 2.218703
             underrated                                 2.217497
       highly recommend                                 2.186460
                  felix                                 2.149143
          heartbreaking                             

## Full corpus

TF-IDF uses unigrams and bigrams, min_df=5, max_df=0.8, max_features=50000 and sublinear term frequencies. Multinomial Naive Bayes uses alpha=0.5.

In [3]:
model, full_metrics = sc.evaluate(usable, "full_corpus", results)


EXPERIMENT: full_corpus train: 39659 test: 9915
Accuracy: 0.8822995461422087 | assignment baseline: 0.500 | improvement: 0.38229954614220873
Training-majority baseline on test: 0.5018658598083712
              precision    recall  f1-score   support

    negative     0.8855    0.8771    0.8813      4939
    positive     0.8792    0.8875    0.8833      4976

    accuracy                         0.8823      9915
   macro avg     0.8823    0.8823    0.8823      9915
weighted avg     0.8823    0.8823    0.8823      9915

Confusion matrix: rows=true, columns=predicted; negative, positive
 [[4332  607]
 [ 560 4416]]

20 most positive terms:
                 term  positive_minus_negative_log_probability
       cast excellent                                 2.663763
   highly recommended                                 2.638025
          goldsworthy                                 2.455127
                felix                                 2.393316
                ponyo                    

## Compare experiments

In [4]:
import pandas as pd
pd.DataFrame([{k: v for k,v in x.items() if k in ["experiment","train_rows","test_rows","accuracy","assignment_baseline"]} for x in [sample_metrics,full_metrics]])

,experiment,train_rows,test_rows,accuracy,assignment_baseline
0,sample_20000,16000,4000,0.88275,0.5
1,full_corpus,39659,9915,0.88230,0.5


## Save the trained pipeline

In [5]:
import joblib
joblib.dump(model, ROOT / "models" / "sentiment_pipeline.joblib")

['C:\\Users\\17div\\Documents\\Codex\\2026-11-11\\ok\\work\\imdb\\models\\sentiment_pipeline.joblib']

## Predict a new review

The saved pipeline expects text already processed by clean_review. These short examples illustrate behavior; they do not establish negation accuracy.

In [6]:
for text in ["This movie is good.", "This movie is not good.", "I never enjoyed this film."]:
    print(text, "=>", model.predict([sc.clean_review(text)])[0])

This movie is good. => positive
This movie is not good. => negative
I never enjoyed this film. => positive


## Limitations

Unigrams and bigrams do not reliably capture sarcasm or negation spanning several words. Evaluation on held-out IMDB reviews also does not establish performance on other languages, product reviews, or newer writing styles.

The model predicts review sentiment, not a film score. Per-film aggregation would require film identifiers, which are absent from this two-column dataset.